# 02 · Pipeline commands (Stages 3–5a)

Turn a typed config into the exact, validated command lines for
RFdiffusionAA, LigandMPNN and ColabFold — and catch the mistakes the
protocol flags before a GPU job ever starts.

In [1]:
import sys, os
# make the package importable from the notebooks/ directory
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
EXAMPLES = os.path.join(ROOT, "examples")
print("project root:", ROOT)


project root: /home/user/biofx_python/enzyme_design


In [2]:
from enzyme_design.contig import build_contig
from enzyme_design.pipeline import (RFdiffusionAAJob, LigandMPNNJob,
                                     ColabFoldJob, consistency_check,
                                     PipelineError)
contig = build_contig([('A', 84, 87)], flank=(10, 120), total_length=(150, 150))

## Stage 3 — RFdiffusionAA

In [3]:
rfd = RFdiffusionAAJob(input_pdb='input/active_site.pdb', contig=contig,
                       ligand='LIG', num_designs=1000, diffuser_T=200,
                       guide_scale=1.0)
print(rfd.to_shell())

apptainer run --nv rf_se3_diffusion.sif -u run_inference.py inference.deterministic=True diffuser.T=200 inference.input_pdb=input/active_site.pdb 'contigmap.contigs=['"'"'10-120,A84-87,10-120'"'"']' 'contigmap.length='"'"'150-150'"'"'' inference.ligand=LIG inference.num_designs=1000 inference.output_prefix=output/run1/sample potentials.guide_scale=1.0


## Stage 4 — LigandMPNN (catalytic residues fixed)

In [4]:
mpnn = LigandMPNNJob(pdb_path='output/run1/sample_0.pdb',
                     fixed_residues=contig.motif_residues(),
                     number_of_batches=8, pack_side_chains=True)
print(mpnn.to_shell(motif_residues=contig.motif_residues()))

python run.py --model_type ligand_mpnn --pdb_path output/run1/sample_0.pdb --fixed_residues 'A84 A85 A86 A87' --number_of_batches 8 --pack_side_chains 1 --out_folder mpnn/run1/


### The CRITICAL STEP, enforced
If a catalytic residue is *not* held fixed, the command builder refuses.

In [5]:
leaky = LigandMPNNJob(pdb_path='s.pdb', fixed_residues=['A84','A85'])  # missing 86,87
try:
    leaky.to_argv(motif_residues=contig.motif_residues())
except PipelineError as e:
    print('blocked:', e)

blocked: motif residues not fixed: ['A86', 'A87'] — catalytic geometry will be lost


## Stage 5a — ColabFold single-sequence reprediction

In [6]:
af = ColabFoldJob(input_fasta='designs.fasta', out_dir='af2/run1/',
                  num_models=5, msa_mode='single_sequence')
print(af.to_shell())

colabfold_batch --num-models 5 --num-recycle 3 --msa-mode single_sequence designs.fasta af2/run1/


## Whole-pipeline cross-check

In [7]:
problems = consistency_check(rfd, mpnn)
print('consistency problems:', problems or 'none — diffusion & design agree')

consistency problems: none — diffusion & design agree
